# Notebook 02: Statistička detekcija financijskih kriza

**Projekt:** Financijske mreže na ZSE (CROBEX, 2004–2026)
**Teme:** Mann-Whitney test, Cohen's d, logistička regresija, BH korekcija višestrukih testiranja
**Kolegiji:** Ekonometrija, Strojno učenje u financijama, Statistika

## Kontekst

Pitanje: **razlikuju li se mrežne mjere statistički između kriznih i mirnih perioda?**

Koristimo tri tehnike:
1. **Mann-Whitney U test** — neparametarski, ne pretpostavlja normalnost
2. **Cohen's d** — veličina efekta (koliko standardnih devijacija razlike?)
3. **Logistička regresija** — može li jedna mjera klasificirati period kao krizni?

Na kraju primjenjujemo **Benjamini-Hochberg (BH) korekciju** za višestruka testiranja.


In [ ]:
# Postavljanje okolisa (automatski detektira Google Colab)
import sys
if "google.colab" in sys.modules:
    import subprocess, os
    subprocess.run(["git", "clone",
                    "https://github.com/svlah-sketch/FinNet-teaching.git", "/content/FinNet-teaching"], check=True)
    os.chdir("/content/FinNet-teaching/notebooks")
    print("Colab: repozitorij kloniran.")
else:
    print("Lokalno pokretanje - nema potrebe za klonom.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

DATA_DIR = Path("../sample_data")

metrics = pd.read_csv(DATA_DIR / "sample_metrics_W90.csv",
                      index_col="window_end", parse_dates=True)

CRISES = {
    "GFC":     ("2007-10-01", "2009-09-30"),
    "EU_DEBT": ("2011-04-01", "2012-09-30"),
    "COVID":   ("2020-02-20", "2021-03-31"),
}

def is_crisis(date):
    for start, end in CRISES.values():
        if pd.Timestamp(start) <= date <= pd.Timestamp(end):
            return True
    return False

metrics["kriza"] = metrics.index.map(is_crisis)
print(f"Kriznih prozora: {metrics['kriza'].sum()}, Mirnih: {(~metrics['kriza']).sum()}")

## 1. Mann-Whitney U test i Cohen's d

Za svaku mjeru testiramo razliku između kriznih i mirnih prozora.

**Cohen's d** interpretacija:
- |d| < 0.2: zanemariv efekt
- |d| ≈ 0.5: srednji efekt
- |d| > 0.8: veliki efekt


In [ ]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*x.std(ddof=1)**2 + (ny-1)*y.std(ddof=1)**2) / (nx+ny-2))
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else np.nan

results = []
metric_cols = [c for c in metrics.columns if c != "kriza"]

for col in metric_cols:
    crisis  = metrics.loc[metrics["kriza"],  col].dropna()
    tranquil = metrics.loc[~metrics["kriza"], col].dropna()
    if len(crisis) < 3 or len(tranquil) < 3:
        continue
    stat, pval = stats.mannwhitneyu(crisis, tranquil, alternative="two-sided")
    d = cohens_d(crisis, tranquil)
    results.append({"mjera": col, "n_kriza": len(crisis), "n_mirno": len(tranquil),
                    "mean_kriza": crisis.mean(), "mean_mirno": tranquil.mean(),
                    "cohen_d": d, "mw_p": pval})

res = pd.DataFrame(results).sort_values("cohen_d", ascending=False)
print(res[["mjera","n_kriza","n_mirno","mean_kriza","mean_mirno","cohen_d","mw_p"]].to_string(index=False))

## 2. Benjamini-Hochberg korekcija

Kad testiramo 10 mjera istovremeno, povećava se rizik lažno pozitivnih nalaza.
BH korekcija kontrolira **stopu lažnih otkrića (FDR)** umjesto individualne p-vrijednosti.


In [ ]:
def bh_correct(pvals, q=0.05):
    m = len(pvals)
    ranked = sorted(enumerate(pvals), key=lambda x: x[1])
    reject = [False] * m
    for rank, (i, p) in enumerate(ranked, 1):
        if p <= q * rank / m:
            reject[i] = True
        else:
            break
    return reject

pvals = res["mw_p"].values
reject = bh_correct(pvals, q=0.05)
res["BH_sig"] = reject

print("BH korekcija (q=0.05):")
print(res[["mjera","cohen_d","mw_p","BH_sig"]].to_string(index=False))

## 3. Logistička regresija: može li mjera predvidjeti krizu?

Koristimo M1 (LCC frakcija) kao prediktor binarne varijable "kriza".


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

best_metric = res.iloc[0]["mjera"]  # najjači Cohen's d
X_raw = metrics[best_metric].values.reshape(-1, 1)
y = metrics["kriza"].astype(int).values

# Izbaci NaN
mask = ~np.isnan(X_raw.ravel())
X_clean, y_clean = X_raw[mask], y[mask]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

model = LogisticRegression()
model.fit(X_scaled, y_clean)
proba = model.predict_proba(X_scaled)[:, 1]
auc = roc_auc_score(y_clean, proba)

fpr, tpr, _ = roc_curve(y_clean, proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f"ROC krivulja (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("Stopa lažno pozitivnih")
plt.ylabel("Stopa istinito pozitivnih")
plt.title(f"ROC krivulja — {best_metric} kao prediktor krize")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("notebook02_roc.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\nPrva mjera: {best_metric}, AUC = {auc:.3f}")

---

## Zadaci za studente

**1.** Mijenjate li `q` u BH korekciji s 0.05 na 0.10, koje dodatne mjere postaju
   statistički značajne? Što to znači za interpretaciju?

**2.** Izračunajte Cohen's d *po krizi* (GFC, EU_DEBT, COVID posebno).
   Je li GFC konzistentno jača kriza od COVID-19 u ovim podacima?

**3.** Zamijenite logistički regresiju s **SVM-om** (`sklearn.svm.SVC` s `probability=True`).
   Usporedite AUC s logističkim modelom. Koji model je bolji? Zašto?

**4.** (Napredni) M4 i M8 su NaN za mnoge prozore. Mogu li se koristiti u
   logističkoj regresiji? Što trebate napraviti s nedostajućim podacima?
